In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)
from sklearn.inspection import permutation_importance
import xgboost as xgb
from xgboost import XGBRegressor, plot_importance
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# SHAP (pre-installed on Kaggle)
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not available – skipping SHAP plots")

# ── Plot style ────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#2e3250",
    "axes.labelcolor":  "#c8cfe8",
    "xtick.color":      "#8892b0",
    "ytick.color":      "#8892b0",
    "text.color":       "#c8cfe8",
    "grid.color":       "#1e2340",
    "grid.linewidth":   0.6,
    "font.family":      "DejaVu Sans",
    "figure.dpi":       120,
})
ACCENT   = "#64ffda"
ACCENT2  = "#f472b6"
ACCENT3  = "#fbbf24"
BLUE     = "#38bdf8"
PURPLE   = "#818cf8"

# ── 1. Load data ──────────────────────────────────────────────
df = pd.read_csv("/kaggle/input/datasets/bapdesilva/betel-ds/betel_yield_dataset_40000_v2.csv")

print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(f"  Shape         : {df.shape}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Target range  : {df['Leaf_Count_per_Bed'].min():.0f} – {df['Leaf_Count_per_Bed'].max():.0f}")
print(f"  Target mean   : {df['Leaf_Count_per_Bed'].mean():.2f} ± {df['Leaf_Count_per_Bed'].std():.2f}")
print("=" * 55)
print(df.describe().round(2))


# ── 2. Feature engineering ────────────────────────────────────
def engineer_features(df):
    df = df.copy()

    # Interaction terms (domain-informed)
    df["Moisture_x_Rain"]    = df["Soil_Moisture"] * df["Rainfall"]
    df["Moisture_x_Humid"]   = df["Soil_Moisture"] * df["Humidity"]
    df["NPK_sum"]            = df["N"] + df["P"] + df["K"]
    df["NPK_product"]        = df["N"] * df["P"] * df["K"]
    df["Temp_x_Humid"]       = df["Temperature"] * df["Humidity"]
    df["Rain_x_Humid"]       = df["Rainfall"] * df["Humidity"]
    df["pH_x_NPK"]           = df["Soil_pH"] * df["NPK_sum"]
    df["Moisture_sq"]        = df["Soil_Moisture"] ** 2
    df["Rain_sq"]            = df["Rainfall"] ** 2

    # Log transforms (reduces skew on rain)
    df["log_Rain"]           = np.log1p(df["Rainfall"])
    df["log_NPK"]            = np.log1p(df["NPK_sum"])

    # Ratios
    df["N_to_P"]             = df["N"] / (df["P"] + 1e-6)
    df["Moisture_to_Rain"]   = df["Soil_Moisture"] / (df["Rainfall"] + 1e-6)

    # Optimal range flags (domain heuristics for betel)
    df["optimal_moisture"]   = ((df["Soil_Moisture"] >= 55) & (df["Soil_Moisture"] <= 75)).astype(int)
    df["optimal_temp"]       = ((df["Temperature"]   >= 25) & (df["Temperature"]   <= 32)).astype(int)
    df["optimal_pH"]         = ((df["Soil_pH"]       >= 5.5) & (df["Soil_pH"]      <= 7.0)).astype(int)
    df["optimal_rain"]       = ((df["Rainfall"]      >= 150) & (df["Rainfall"]     <= 300)).astype(int)

    return df

df_eng = engineer_features(df)

FEATURES = [c for c in df_eng.columns if c != "Leaf_Count_per_Bed"]
TARGET   = "Leaf_Count_per_Bed"

X = df_eng[FEATURES]
y = df_eng[TARGET]

print(f"\n  Total features after engineering: {len(FEATURES)}")
